# FINRL Walk-Forward Experiment

This notebook runs the direct-feature walk-forward experiment runner and visualizes portfolio performance against the S&P 500 / SPY benchmark.

Use small synthetic data locally. Use Colab for the full configured universe, direct portfolio optimization, and full walk-forward experiments.

In [ ]:
%cd /content/FINRL
!git clone https://github.com/nidarshans/FINRL.git

In [ ]:
%cd /content/FINRL/src
#%pip install -e .


In [1]:
# Keep notebook imports pointed at the live workspace package.

from datetime import date, timedelta

import polars as pl

from finrl.backtest.walk_forward import WalkForwardConfig
from finrl.dpo_jax import DPOConfig
from finrl.data import (
    MarketDataBundle,
    MarketDataConfig,
    UniverseConfig,
    build_rebalance_calendar,
    compute_open_to_open_returns,
    download_ohlcv,
)
from finrl.data.download import download_macro_series
from finrl.env.trading_env import EnvConfig
from finrl.experiments import (
    ExperimentConfig,
    RawExperimentData,
    build_allocation_figure,
    build_holdings_heatmap_granular,
    build_performance_figure,
    build_regime_portfolio_figure,
    build_spectral_figure,
    metrics_to_frame,
    run_walk_forward_experiment,
)
from finrl.features import FeatureConfig, build_feature_bundle, selected_direct_allocation_indices
from finrl.features.preprocessing import PreprocessingConfig
from finrl.features.schema import FeatureBundle

## Prepared Data Contract

The runner expects prepared feature and return tables:

- `FeatureBundle` with asset and macro features plus a dummy 20-column spectral compatibility table.
- `returns`: Polars DataFrame with `decision_date` and one return column per tradable asset.
- `spy_returns`: Polars DataFrame with `decision_date` and `spy_return` for the same holding periods.

Replace the synthetic fixture below with the output of the data, feature, preprocessing, and return-preparation pipeline for full experiments. DPO trains the direct allocation head over explicitly routed asset features.

## Run With Real yfinance Data

Edit `TICKERS`, `START`, `END`, and `MAX_STOCKS`, then run this section in Colab. The code downloads real stock or bond ticker data plus SPY, computes daily open-to-open returns, builds causal per-asset features, and packages everything into `RawExperimentData` for the walk-forward runner.

In [28]:
TICKERS = [
    "XLC", "XLY", "XLP", "XLE", "XLF",
    "XLV", "XLI", "XLB", "XLRE", "XLK", "XLU",
    "BIL", "IAU", "SLV", "XBI", "SPY"
]
MAX_STOCKS = len(TICKERS)  # set to 100 after pasting your full universe
START = "2017-01-01"
END = "2026-07-25"
CACHE_DIR = "data/cache"
BENCHMARK_TICKER = "SPY"
REBALANCE_FREQUENCY = "daily"  # "daily" or "weekly"

universe = UniverseConfig(
    tickers=TICKERS,
    max_stocks=MAX_STOCKS,
    include_cash=False,
    benchmark_ticker=BENCHMARK_TICKER,
)
market_config = MarketDataConfig(
    universe=universe,
    start=START,
    end=END,
    cache_dir=CACHE_DIR,
)
selected_tickers = universe.selected_tickers
selected_tickers

('XLC',
 'XLY',
 'XLP',
 'XLE',
 'XLF',
 'XLV',
 'XLI',
 'XLB',
 'XLRE',
 'XLK',
 'XLU',
 'BIL',
 'IAU',
 'SLV',
 'XBI',
 'SPY')

In [29]:
def _returns_wide(open_to_open_returns: pl.DataFrame, tickers: tuple[str, ...]) -> pl.DataFrame:
    wide = (
        open_to_open_returns
        .select(["decision_date", "ticker", "return"])
        .pivot(index="decision_date", on="ticker", values="return", aggregate_function="first")
        .sort("decision_date")
    )
    return wide.select(["decision_date", *tickers]).drop_nulls()


def _spy_returns(open_to_open_returns: pl.DataFrame) -> pl.DataFrame:
    return (
        open_to_open_returns
        .select(["decision_date", pl.col("return").alias("spy_return")])
        .sort("decision_date")
        .drop_nulls()
    )


def _filter_features_to_common_dates(features: FeatureBundle, returns: pl.DataFrame, spy_returns: pl.DataFrame) -> FeatureBundle:
    common_dates = (
        returns.select("decision_date")
        .join(spy_returns.select("decision_date"), on="decision_date", how="inner")
        .rename({"decision_date": "date"})
        .with_columns(pl.col("date").cast(pl.Date))
        .unique()
        .sort("date")
    )
    asset = features.asset_features.join(common_dates, on="date", how="inner").sort(["date", "ticker"])
    macro = (
        common_dates
        .join(features.macro_features, on="date", how="left")
        .sort("date")
        .with_columns(pl.all().exclude("date").forward_fill().fill_null(0.0))
    )
    spectral = features.spectral_features.join(common_dates, on="date", how="inner").sort("date")
    dates = tuple(common_dates.get_column("date").to_list())
    return FeatureBundle(
        asset_features=asset,
        macro_features=macro,
        spectral_features=spectral,
        decision_dates=dates,
        tickers=features.tickers,
        asset_feature_columns=features.asset_feature_columns,
        macro_feature_columns=features.macro_feature_columns,
        spectral_feature_columns=features.spectral_feature_columns,
    )


def make_real_yfinance_data() -> RawExperimentData:
    ohlcv = download_ohlcv(selected_tickers, START, END, market_config)
    spy_ohlcv = download_ohlcv((BENCHMARK_TICKER,), START, END, market_config)
    macro = download_macro_series(START, END, market_config)
    calendar = build_rebalance_calendar(ohlcv, REBALANCE_FREQUENCY)

    market_bundle = MarketDataBundle(
        ohlcv=ohlcv,
        spy_ohlcv=spy_ohlcv,
        macro=macro,
        calendar=calendar,
    )
    features = build_feature_bundle(
        market_bundle,
        FeatureConfig(
            accumulation_window=40,
            klinger_fast_span=34,
            klinger_slow_span=55,
            klinger_signal_span=13,
            macd_fast_span=12,
            macd_slow_span=26,
            macd_signal_span=9,
            mr_ewma_span=50,
            mr_vol_window=50,
            spectral_dim=20,
            use_spectral_features=False,
            include_hawkes=False,
        ),
    )

    stock_returns = _returns_wide(
        compute_open_to_open_returns(ohlcv, calendar),
        selected_tickers,
    )
    spy_returns = _spy_returns(compute_open_to_open_returns(spy_ohlcv, calendar))
    features = _filter_features_to_common_dates(features, stock_returns, spy_returns)
    common_dates = pl.DataFrame({"decision_date": list(features.decision_dates)}).with_columns(pl.col("decision_date").cast(pl.Date))
    stock_returns = common_dates.join(stock_returns, on="decision_date", how="inner")
    spy_returns = common_dates.join(spy_returns, on="decision_date", how="inner")
    return RawExperimentData(features=features, returns=stock_returns, spy_returns=spy_returns)


raw_data = make_real_yfinance_data()
raw_data.features.asset_features.tail(), raw_data.returns.tail(), raw_data.spy_returns.tail()

(shape: (5, 64)
 ┌────────────┬────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ date       ┆ ticker ┆ open       ┆ high      ┆ … ┆ liq_liqui ┆ return_pe ┆ dollar_vo ┆ amihud_pe │
 │ ---        ┆ ---    ┆ ---        ┆ ---       ┆   ┆ dity_shoc ┆ rcentile_ ┆ lume_perc ┆ rcentile_ │
 │ date       ┆ str    ┆ f64        ┆ f64       ┆   ┆ k         ┆ rank      ┆ entile_ra ┆ rank      │
 │            ┆        ┆            ┆           ┆   ┆ ---       ┆ ---       ┆ nk        ┆ ---       │
 │            ┆        ┆            ┆           ┆   ┆ f64       ┆ f64       ┆ ---       ┆ f64       │
 │            ┆        ┆            ┆           ┆   ┆           ┆           ┆ f64       ┆           │
 ╞════════════╪════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ 2026-06-30 ┆ XLP    ┆ 83.989998  ┆ 84.089996 ┆ … ┆ -0.073223 ┆ 0.066667  ┆ 0.466667  ┆ 0.4       │
 │ 2026-06-30 ┆ XLRE   ┆ 44.375     ┆ 44.52     ┆ … ┆ -0.805679 ┆ 

In [60]:
n_stocks = len(raw_data.features.tickers)
n_tradable_assets = n_stocks + 1  # risky assets plus cash
asset_feature_dim = len(raw_data.features.asset_feature_columns)
macro_feature_dim = len(raw_data.features.macro_feature_columns)
routing = selected_direct_allocation_indices(raw_data.features.asset_feature_columns)

# Learned policies emit dense weights. The environment can then keep only the
# top N risky positions, preserve cash, and renormalize the executed portfolio.
# Set to None to trade the full dense allocation.
POLICY_MODE = "dpo"  # "dpo" or "equal_weight"
TOP_N_POSITIONS = 20

config = ExperimentConfig(
    walk_forward=WalkForwardConfig(train_years=2, test_years=1, step_years=1),
    preprocessing=PreprocessingConfig(rolling_window=252),
    dpo=DPOConfig(
        learning_rate=3e-4,
        num_epochs=50,
        batch_size=32,
        transaction_cost_bps=10.0,
        lambda_turnover=0.0,
        lambda_drawdown=0.0,
        lambda_concentration=0.0,
        allocation_hidden_dims=(8, 16, 8, 4),
        allocation_hidden_activation="tanh",
        allocation_output_activation="identity",
        allocation_use_layer_norm=True,
        simplex_activation="sparsemax",
    ),
    env=EnvConfig(
        sortino_target_return=0.0,
        sortino_downside_penalty=0.0,
        top_n_positions=TOP_N_POSITIONS,
    ),
    enable_dpo=POLICY_MODE == "dpo",
    rebalance_frequency=REBALANCE_FREQUENCY,
    seed=5,
)

print({
    "stocks": n_stocks,
    "tradable_assets": n_tradable_assets,
    "asset_feature_dim": asset_feature_dim,
    "macro_feature_dim": macro_feature_dim,
    "policy_mode": POLICY_MODE,
    "top_n_positions": TOP_N_POSITIONS,
    "direct_features": len(routing.direct_allocation_indices),
    "decision_dates": len(raw_data.features.decision_dates),
})

result = run_walk_forward_experiment(raw_data, config)
metrics_to_frame(result)

{'stocks': 16, 'tradable_assets': 17, 'asset_feature_dim': 62, 'macro_feature_dim': 24, 'policy_mode': 'dpo', 'top_n_positions': 20, 'direct_features': 6, 'decision_dates': 2019}


split_index,test_start,test_end,portfolio_cumulative_return,spy_cumulative_return,spy_relative_alpha,portfolio_max_drawdown,portfolio_mean_turnover,portfolio_total_transaction_cost
i64,date,date,f64,f64,f64,f64,f64,f64
0,2020-01-01,2020-12-31,0.369424,0.146158,0.223267,0.219913,0.22366,0.056586
1,2021-01-01,2021-12-31,0.072213,0.301874,-0.229661,0.059871,0.252777,0.0637
2,2022-01-01,2022-12-31,-0.088494,-0.200409,0.111915,0.239042,0.24037,0.060333
3,2023-01-01,2023-12-31,-0.006531,0.2277,-0.234231,0.115805,0.336405,0.084101
4,2024-01-01,2024-12-31,0.083103,0.248921,-0.165819,0.054458,0.182998,0.046115
5,2025-01-01,2025-12-31,0.149364,0.168519,-0.019155,0.153373,0.240192,0.060048
6,2026-01-01,2026-12-31,0.05402,0.085152,-0.031131,0.124583,0.202138,0.024661


## Performance vs S&P 500


In [61]:
performance_fig = build_performance_figure(result)
performance_fig.show()


## Portfolio Allocation


In [62]:
allocation_fig = build_allocation_figure(result)
allocation_fig.show()


## Holdings Heatmap


In [63]:
holdings_heatmap_fig = build_holdings_heatmap_granular(
    result,
    min_weight=0.001,
    top_n=50,
    freq=None,
    include_cash=False,
)
holdings_heatmap_fig.show()


## Regime Portfolio


In [64]:
regime_portfolio_fig = build_regime_portfolio_figure(result)
regime_portfolio_fig.show()


In [65]:
# Optional report export
#from finrl.experiments import write_report
#write_report(result, "walk_forward_report")